# E-Commerce Sales Data Analysis — Exploratory Data Analysis

## Week 6 Internship Assignment
**Project:** E-Commerce Sales Data Analysis and Interactive Power BI Dashboard  
**Student:** [Your Name]  
**Date:** October 2026

> **⚠️ Data Notice:** This project uses **synthetically generated** data that simulates a retail superstore dataset. All findings are educational and do not represent real business data.

---

## A. Business Introduction

### Project Overview
This project performs an end-to-end analysis of retail e-commerce sales data to derive actionable business insights. The analysis covers sales performance, product profitability, customer segmentation, regional analysis, and the impact of discounting on profitability.

### Problem Statement
E-commerce businesses need to understand:
1. Which products and categories drive the most revenue and profit?
2. Which customer segments and regions are most valuable?
3. How do discounts affect profitability?
4. What seasonal patterns exist in sales data?

### Objectives
1. Analyze total sales and profit performance across categories, regions, and segments
2. Identify high-performing and low-performing products
3. Investigate monthly and yearly sales trends
4. Examine the discount-profitability relationship
5. Generate evidence-based business insights

### Dataset Source
- **Type:** Synthetic dataset modeled after the Tableau Sample Superstore
- **Generated by:** `src/generate_dataset.py` (reproducible with seed=42)
- **Records:** 9,994 (after cleaning)
- **Time Period:** January 2020 – December 2023

### Business Questions
1. What are the total sales, profit, and profit margin for the entire period?
2. Which category contributes the most to revenue and profit?
3. Which region is the most profitable?
4. How does the Consumer, Corporate, and Home Office segment compare?
5. What is the relationship between discounts and profit?
6. Which products generate the most revenue?
7. Are there seasonal patterns in sales?


## Setup: Import Libraries and Load Data

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100

# Load cleaned dataset
BASE_DIR = os.path.dirname(os.getcwd())  # notebooks/ -> project root
CLEAN_PATH = os.path.join(BASE_DIR, 'data', 'cleaned_sales.csv')

df = pd.read_csv(CLEAN_PATH)
df['Order Date'] = pd.to_datetime(df['Order Date'])
if 'Ship Date' in df.columns:
    df['Ship Date'] = pd.to_datetime(df['Ship Date'])

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")


## B. Data Inspection

In [ ]:
# First 5 records
df.head()

In [ ]:
# Last 5 records
df.tail()

In [ ]:
# Dataset shape
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

In [ ]:
# Column names
print('Columns:', list(df.columns))

In [ ]:
# Data types
df.dtypes

In [ ]:
# Missing value report
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("Columns with missing values:")
print(missing_report[missing_report['Missing Count'] > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum():,}")


In [ ]:
# Duplicate count
n_dups = df.duplicated().sum()
print(f'Duplicate rows in cleaned dataset: {n_dups}')


## C. Descriptive Statistics

In [ ]:
# Summary statistics for numerical columns
num_cols = ['Sales', 'Profit', 'Quantity', 'Discount', 'Profit Margin %']
stats = df[num_cols].describe().T
stats['median'] = df[num_cols].median()
stats.round(2)


In [ ]:
# Frequency tables
for col in ['Category', 'Segment', 'Region', 'Ship Mode']:
    print(f"\n{col} Distribution:")
    freq = df[col].value_counts()
    pct = (freq / len(df) * 100).round(1)
    print(pd.DataFrame({'Count': freq, 'Percentage': pct}))


## D. Data Quality Analysis

In [ ]:
# Outlier analysis using IQR method
for col in ['Sales', 'Profit']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers):,} potential outliers "
          f"(below {lower:.2f} or above {upper:.2f})")
    print(f"  Decision: RETAIN — these represent legitimate bulk orders")


## E. Business Analysis

In [ ]:
# Key Performance Indicators
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_qty = df['Quantity'].sum()
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()
profit_margin = total_profit / total_sales * 100

print("=" * 50)
print("KEY PERFORMANCE INDICATORS")
print("=" * 50)
print(f"  Total Sales:        ${total_sales:>15,.2f}")
print(f"  Total Profit:       ${total_profit:>15,.2f}")
print(f"  Total Quantity:     {total_qty:>16,} units")
print(f"  Total Orders:       {total_orders:>16,}")
print(f"  Unique Customers:   {total_customers:>16,}")
print(f"  Profit Margin:      {profit_margin:>15.2f}%")
print(f"  Avg Order Value:    ${total_sales/total_orders:>15,.2f}")


In [ ]:
# Sales by Category
cat_analysis = df.groupby('Category').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Transactions=('Sales', 'count'),
    Avg_Order_Value=('Sales', 'mean')
).round(2)
cat_analysis['Profit_Margin_%'] = (cat_analysis['Total_Profit'] / cat_analysis['Total_Sales'] * 100).round(2)
cat_analysis.sort_values('Total_Sales', ascending=False)


In [ ]:
# Sales by Region
region_analysis = df.groupby('Region').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Transactions=('Sales', 'count')
).round(2)
region_analysis['Profit_Margin_%'] = (region_analysis['Total_Profit'] / region_analysis['Total_Sales'] * 100).round(2)
region_analysis.sort_values('Total_Sales', ascending=False)


In [ ]:
# Customer Segment Analysis
seg_analysis = df.groupby('Segment').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Transactions=('Sales', 'count'),
    Unique_Customers=('Customer ID', 'nunique')
).round(2)
seg_analysis['Profit_Margin_%'] = (seg_analysis['Total_Profit'] / seg_analysis['Total_Sales'] * 100).round(2)
seg_analysis.sort_values('Total_Sales', ascending=False)


In [ ]:
# Monthly Sales Trend
monthly = df.groupby(['Order Year', 'Order Month']).agg(
    Monthly_Sales=('Sales', 'sum'),
    Monthly_Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique')
).reset_index()
print("Monthly sales stats:")
print(f"  Highest month: ${monthly['Monthly_Sales'].max():,.2f}")
print(f"  Lowest month:  ${monthly['Monthly_Sales'].min():,.2f}")
print(f"  Average month: ${monthly['Monthly_Sales'].mean():,.2f}")


In [ ]:
# Top 10 Products by Sales
top10 = df.groupby('Product Name')['Sales'].sum().nlargest(10)
print("Top 10 Products by Sales:")
for i, (prod, sales) in enumerate(top10.items(), 1):
    print(f"  {i:2d}. {prod[:45]:<45} ${sales:>10,.2f}")


In [ ]:
# Sub-Category Analysis (Sales and Profit)
subcat = df.groupby('Sub-Category').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).round(2)
subcat['Profit_Margin_%'] = (subcat['Total_Profit'] / subcat['Total_Sales'] * 100).round(2)
subcat_sorted = subcat.sort_values('Total_Sales', ascending=False)
print("Sub-Category Performance (sorted by Sales):")
print(subcat_sorted.to_string())


In [ ]:
# Discount vs Profit Analysis
discount_bins = pd.cut(df['Discount'], bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
                       labels=['No discount','1-10%','11-20%','21-30%','31-50%','51%+'])
disc_analysis = df.groupby(discount_bins, observed=True).agg(
    Avg_Profit=('Profit', 'mean'),
    Transactions=('Profit', 'count'),
    Loss_Transactions=('Profit', lambda x: (x < 0).sum())
).round(2)
disc_analysis['Loss_Rate_%'] = (disc_analysis['Loss_Transactions'] / disc_analysis['Transactions'] * 100).round(1)
print("Discount Band Analysis:")
print(disc_analysis.to_string())


## F. Correlation Analysis

In [ ]:
# Correlation matrix
num_cols = ['Sales', 'Profit', 'Quantity', 'Discount', 'Profit Margin %']
corr_matrix = df[num_cols].corr()
print("Correlation Matrix:")
print(corr_matrix.round(3))
print()
print("Key insights:")
print(f"  Sales-Profit correlation:          {corr_matrix.loc['Sales','Profit']:.3f} (strong positive)")
print(f"  Discount-Profit correlation:       {corr_matrix.loc['Discount','Profit']:.3f} (negative)")
print(f"  Discount-ProfitMargin correlation: {corr_matrix.loc['Discount','Profit Margin %']:.3f} (strong negative)")


In [ ]:
# Display correlation heatmap
from IPython.display import Image
import os
viz_dir = os.path.join(os.path.dirname(os.getcwd()), 'visualizations')
img_path = os.path.join(viz_dir, '06_correlation_heatmap.png')
if os.path.exists(img_path):
    display(Image(img_path, width=700))


## G. Data Visualizations

The following charts were generated by `src/eda_analysis.py` and saved to `visualizations/`.

In [ ]:
from IPython.display import Image, display
import os
viz_dir = os.path.join(os.path.dirname(os.getcwd()), 'visualizations')

charts = [
    ('01_sales_by_category.png', 'Chart 1: Sales by Category'),
    ('02_monthly_sales_trend.png', 'Chart 2: Monthly Sales Trend'),
    ('03_profit_by_region.png', 'Chart 3: Profit by Region'),
    ('04_sales_by_segment.png', 'Chart 4: Sales by Customer Segment'),
    ('05_discount_vs_profit.png', 'Chart 5: Discount vs Profit'),
    ('06_correlation_heatmap.png', 'Chart 6: Correlation Heatmap'),
    ('07_top10_products.png', 'Chart 7: Top 10 Products by Sales'),
    ('08_sales_profit_subcategory.png', 'Chart 8: Sales & Profit by Sub-Category'),
    ('09_yearly_comparison.png', 'Chart 9: Year-over-Year Comparison'),
    ('10_profit_distribution.png', 'Chart 10: Profit Distribution'),
]

for fname, title in charts:
    path = os.path.join(viz_dir, fname)
    if os.path.exists(path):
        print(f"\n### {title}")
        display(Image(path, width=750))
    else:
        print(f"[MISSING] {fname} - run src/eda_analysis.py to generate")


## H. Key Findings

Based on analysis of 9,994 synthetic sales transactions (Jan 2020 – Dec 2023):

### 1. Revenue Performance
- **Total Sales:** \$33,170,959.70 across 9,070 unique orders
- **Total Profit:** \$3,642,648.41 (10.98% overall margin)
- **Average Order Value:** \$3,657.22

### 2. Category Analysis
- Technology leads in transactions (3,389 records) and average sale value
- Office Supplies has the most consistent (moderate) profitability
- Furniture has the most variable profitability — Tables sub-category is loss-making

### 3. Regional Performance
- All 4 regions (East, West, Central, South) generate positive profit
- West and East lead by transaction volume
- Central and South have growth opportunities relative to their market size

### 4. Customer Segments
- Consumer: 52.1% of transactions (largest volume, B2C)
- Corporate: 30.0% (higher-value bulk orders, B2B)
- Home Office: 18.0% (small, growth segment)

### 5. Discount-Profit Relationship
- **Strongest finding:** Discount ↔ Profit Margin correlation = **-0.854**
- Discounts above 20% frequently produce loss-making transactions
- Tables sub-category has structural margin problems independent of discounts

### 6. Seasonal Patterns
- Q4 (Oct–Dec) consistently generates peak monthly sales
- January and February are the weakest months
- The pattern repeats across all 4 years, confirming predictable seasonality

### 7. Outlier Analysis
- 1,733 potential profit outliers detected (±3×IQR)
- These represent legitimate bulk/high-value orders, not data errors
- Decision: **Retained** — removing them would distort aggregate analysis

> **Note:** All findings are derived from synthetic data for educational purposes only.
